# 03 — Preprocessing: Normalization & Patch Extraction

Turns the raw Alps elevation map (DEM) into model-ready training data:
elevations are rescaled to [-1, 1], then the map is cut into 256×256 patches.

## 1. Setup

In [1]:
import os, json
import numpy as np
import rasterio
import matplotlib.pyplot as plt 

## 2. Load the DEM

Read the elevation band from the GeoTIFF into a NumPy array.
Convert to float32 because normalization produces decimal values.

In [2]:
src = rasterio.open("../data/output_be.tif")   # open the Alps DEM
elev = src.read(1).astype(np.float32)           # band 1 = elevation in meters
print(elev.shape)

(4545, 6763)


## 3. Normalization

Diffusion models train stably only on small, centered inputs, so we
rescale elevations (meters) to [-1, 1]. `denormalize` is the inverse —
it converts model output back to meters for use in Unreal.

In [3]:
ELEV_MIN = -500   # project-wide elevation floor (meters)
ELEV_MAX = 4500   # project-wide elevation ceiling (meters)

def normalize(x):
    # shift to zero, then compress to [0, 1]
    res = (x - ELEV_MIN) / (ELEV_MAX - ELEV_MIN)
    # stretch [0, 1] to [-1, 1]
    res = res * 2 - 1
    return res

def denormalize(y):
    # reverse of normalize: [-1, 1] back to meters
    res = (y + 1) / 2
    res = res * (ELEV_MAX - ELEV_MIN) + ELEV_MIN
    return res

In [4]:
norm = normalize(elev)          # normalize the whole map at once
print(norm.min(), norm.max())   # sanity check: values fall within [-1, 1]

-0.8024 0.794


## 4. Patch extraction

The full map is too large to feed to a model, so we slide a 256×256
window across it and save each tile as one training sample. Non-overlapping
tiles give a 26×17 grid = 442 patches.

In [5]:
PATCH_SIZE = 256        # side length of each square patch (pixels)

H, W = norm.shape       # map height (rows) and width (columns)
n_x = W // PATCH_SIZE   # how many full patches fit across
n_y = H // PATCH_SIZE   # how many fit down
total = n_x * n_y       # total number of patches
print(n_x, n_y, total)

26 17 442


In [6]:
os.makedirs("../outputs/processed/patches", exist_ok=True)   # create output folder

count = 0
for j in range(n_y):            # loop over patch rows
    for i in range(n_x):        # loop over patch columns
        y0 = j * PATCH_SIZE     # top pixel of this patch
        x0 = i * PATCH_SIZE     # left pixel of this patch
        patch = norm[y0:y0+PATCH_SIZE, x0:x0+PATCH_SIZE]   # cut the tile
        np.save(f"../outputs/processed/patches/patch_{count:04d}.npy", patch)
        count += 1              # move to the next patch number
print("saved", count, "patches")

saved 442 patches


## 5. Save parameters

Record the normalization range and patch settings so model output can
later be converted back to meters, and so the run is reproducible.

In [7]:
params = {
    "elev_min": ELEV_MIN,
    "elev_max": ELEV_MAX,
    "patch_size": PATCH_SIZE,
    "n_patches": total,
}
with open("../outputs/processed/params.json", "w") as f:
    json.dump(params, f, indent=2)   # write settings as readable JSON
print("saved params")

saved params
